# Bioinformatics Exercise 10 — PBIO (s31880)

Adjust **paths** in the first cells if your files live elsewhere:

- `GDS2490.soft`
- `sample_data.fastq` or `sample_data.fastq.gz`

Figures are saved as PNG files next to this notebook.

## Task 1 — Gene expression (GDS2490)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

GDS_PATH = "/Users/maxxxxx/Downloads/GDS2490.soft"


def load_gds2490_soft(path: str) -> pd.DataFrame:
    with open(path, encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("!Dataset_table_begin") or line.startswith(
                "!dataset_table_begin"
            ):
                break
        header = handle.readline().rstrip("\n").split("\t")
        rows = []
        for line in handle:
            if line.startswith("!Dataset_table_end") or line.startswith("!dataset_table_end"):
                break
            if line.startswith("#") or line.startswith("^"):
                continue
            rows.append(line.rstrip("\n").split("\t"))
    df = pd.DataFrame(rows, columns=header)
    sample_cols = [c for c in df.columns if c.startswith("GSM")]
    for c in sample_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.rename(columns={"IDENTIFIER": "gene"})
    return df


df_wide = load_gds2490_soft(GDS_PATH)
NON_SMOKER = ["GSM114084", "GSM114085", "GSM114086", "GSM114087", "GSM114088"]
SMOKER = ["GSM114078", "GSM114079", "GSM114080", "GSM114081", "GSM114082", "GSM114083"]

print(f"Loaded data: {len(df_wide)} genes, {len(NON_SMOKER + SMOKER)} samples")
print(f"Groups: non-smoker ({len(NON_SMOKER)}), smoker ({len(SMOKER)})")


In [ ]:
long_df = df_wide.melt(
    id_vars=["gene"],
    value_vars=NON_SMOKER + SMOKER,
    var_name="sample",
    value_name="expression",
)
long_df["group"] = long_df["sample"].map(
    lambda s: "non-smoker" if s in NON_SMOKER else "smoker"
)
long_df.head()


In [ ]:
means = long_df.groupby(["gene", "group"])["expression"].mean().unstack()
means.columns = ["non-smoker", "smoker"]
means["abs_diff"] = (means["smoker"] - means["non-smoker"]).abs()
top10 = means.nlargest(10, "abs_diff")

summary = []
for gene in top10.index:
    ns = float(means.loc[gene, "non-smoker"])
    sm = float(means.loc[gene, "smoker"])
    summary.append(
        {
            "gene": gene,
            "mean_non_smoker": ns,
            "mean_smoker": sm,
            "difference_smoker_minus_ns": sm - ns,
        }
    )

stats_tbl = pd.DataFrame(summary)
print("\nTop 10 genes with greatest expression difference:\n")
print(stats_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))


In [ ]:
sns.set_theme(style="whitegrid")
top_genes = top10.index.tolist()
sub = long_df[long_df["gene"].isin(top_genes)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=sub, x="gene", y="expression", hue="group")
plt.xticks(rotation=45, ha="right")
plt.title("Top 10 genes — boxplot")
plt.tight_layout()
plt.savefig("boxplot_top10_genes.png", dpi=150)
plt.show()

plt.figure(figsize=(12, 6))
sns.violinplot(data=sub, x="gene", y="expression", hue="group", split=True)
plt.xticks(rotation=45, ha="right")
plt.title("Top 10 genes — violinplot")
plt.tight_layout()
plt.savefig("violinplot_top10_genes.png", dpi=150)
plt.show()

ordered_samples = NON_SMOKER + SMOKER
heat = df_wide.set_index("gene").loc[top_genes, ordered_samples]
plt.figure(figsize=(10, 8))
sns.heatmap(heat, cmap="viridis")
plt.title("Heatmap — samples grouped: non-smoker | smoker")
plt.tight_layout()
plt.savefig("heatmap_top10_genes.png", dpi=150)
plt.show()

print(
    "Saved: boxplot_top10_genes.png, violinplot_top10_genes.png, heatmap_top10_genes.png"
)


## Task 2 — Wisconsin breast cancer classification

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="diagnosis")  # 0=malignant, 1=benign

print("=== WISCONSIN BREAST CANCER — CLASSIFICATION ===")
print(f"Number of samples: {len(X)}")
print(f"Number of features: {X.shape[1]}")
print("Class distribution:")
print(f"  Benign (1): {(y == 1).sum()} ({100 * (y == 1).mean():.1f}%)")
print(f"  Malignant (0): {(y == 0).sum()} ({100 * (y == 0).mean():.1f}%)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nData split: {len(X_train)} training, {len(X_test)} test")
print("Class distribution in training set:")
print(f"  Benign: {100 * (y_train == 1).mean():.1f}%")
print(f"  Malignant: {100 * (y_train == 0).mean():.1f}%")
print("Class distribution in test set:")
print(f"  Benign: {100 * (y_test == 1).mean():.1f}%")
print(f"  Malignant: {100 * (y_test == 0).mean():.1f}%")


In [ ]:
def eval_model(name: str, model, png_name: str):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
    print(f"Precision: {precision_score(y_test, pred):.4f}")
    print(f"Recall:    {recall_score(y_test, pred):.4f}")
    print(f"F1-score:  {f1_score(y_test, pred):.4f}")
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Malignant (0)", "Benign (1)"],
        yticklabels=["Malignant (0)", "Benign (1)"],
    )
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.title(name)
    plt.tight_layout()
    plt.savefig(png_name, dpi=150)
    plt.show()
    print(f"Confusion matrix saved: {png_name}")


eval_model(
    "LOGISTIC REGRESSION",
    LogisticRegression(max_iter=10000),
    "confusion_matrix_lr.png",
)
eval_model(
    "RANDOM FOREST",
    RandomForestClassifier(n_estimators=100, random_state=42),
    "confusion_matrix_rf.png",
)


## Task 3 — FASTQ QC / per-base quality

In [ ]:
import gzip
from pathlib import Path
import numpy as np

FASTQ_PATH = Path("/Users/maxxxxx/Downloads/sample_data.fastq")


def parse_fastq(filepath: Path):
    opener = gzip.open if str(filepath).endswith(".gz") else open
    mode = "rt" if str(filepath).endswith(".gz") else "r"
    with opener(filepath, mode, encoding="ascii", errors="replace") as handle:
        while True:
            header = handle.readline()
            if not header:
                break
            seq = handle.readline().strip()
            handle.readline()
            qual = handle.readline().strip()
            yield seq, qual


def quality_to_phred(quality_string: str) -> list[int]:
    return [ord(char) - 33 for char in quality_string]


sequences, read_mean_q, all_phreds = [], [], []
for seq, qual in parse_fastq(FASTQ_PATH):
    phreds = quality_to_phred(qual)
    sequences.append(seq)
    read_mean_q.append(float(np.mean(phreds)))
    all_phreds.extend(phreds)

lengths = [len(s) for s in sequences]
n_reads = len(sequences)

print(f"File: {FASTQ_PATH.name}")
print(f"Number of reads: {n_reads}")
if min(lengths) == max(lengths):
    print(f"Read length: {min(lengths)} bp (constant)")
else:
    print(
        f"Read length (variable): min={min(lengths)}, max={max(lengths)}, "
        f"mean={np.mean(lengths):.2f} bp"
    )
print("\nQuality statistics:")
print(f"  Mean quality (all nucleotides): {np.mean(all_phreds):.2f}")
print(
    f"  Reads with mean Q > 30: {sum(m > 30 for m in read_mean_q)} "
    f"({100 * sum(m > 30 for m in read_mean_q) / n_reads:.1f}%)"
)
print(
    f"  Reads with mean Q > 20: {sum(m > 20 for m in read_mean_q)} "
    f"({100 * sum(m > 20 for m in read_mean_q) / n_reads:.1f}%)"
)


In [ ]:
import matplotlib.patches as mpatches

qualities_by_pos: list[list[int]] = []
for seq, qual in parse_fastq(FASTQ_PATH):
    phreds = quality_to_phred(qual)
    if len(qualities_by_pos) < len(phreds):
        qualities_by_pos.extend(
            [[] for _ in range(len(phreds) - len(qualities_by_pos))]
        )
    for i, q in enumerate(phreds):
        qualities_by_pos[i].append(q)

positions = np.arange(1, len(qualities_by_pos) + 1)
data = qualities_by_pos

fig, ax = plt.subplots(figsize=(14, 5))
ymax = 40
ax.axhspan(28, ymax, facecolor="#90EE90", alpha=0.35, zorder=0)
ax.axhspan(20, 28, facecolor="#FFD580", alpha=0.35, zorder=0)
ax.axhspan(0, 20, facecolor="#FFB6C1", alpha=0.35, zorder=0)

bp = ax.boxplot(
    data, positions=positions, widths=0.6, showfliers=False, patch_artist=True
)
for patch in bp["boxes"]:
    patch.set_facecolor("#4a90d9")
    patch.set_alpha(0.7)

ax.set_xlim(0.5, len(positions) + 0.5)
ax.set_ylim(0, ymax)
ax.set_xlabel("Position in read (bp)")
ax.set_ylabel("Phred quality score")
ax.set_title("Per base sequence quality")
green = mpatches.Patch(color="#90EE90", alpha=0.5, label="Good (Q > 28)")
orange = mpatches.Patch(color="#FFD580", alpha=0.5, label="Medium (20–28)")
red = mpatches.Patch(color="#FFB6C1", alpha=0.5, label="Poor (Q < 20)")
ax.legend(handles=[green, orange, red], loc="lower right")
plt.tight_layout()
plt.savefig("per_base_quality.png", dpi=150)
plt.show()

